# PLS Regression — estudo conceitual

**Etapa:** fundamentação do modelo especialista

Apresentamos a intuição dos componentes latentes, a relação com o alvo e o uso do PLS sob multicolinearidade.

**Entrada:** protocolo e síntese técnica do PLS  
**Resultado principal:** fundamentos, hipóteses, vantagens e limitações organizados


## 1. Importações, caminhos e funções

In [1]:
from pathlib import Path
import hashlib, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from IPython.display import display

warnings.filterwarnings("ignore")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "bases/grupo5/gold_daily_modeling.csv").exists())
BASE_PATH = ROOT / "bases/grupo5/gold_daily_modeling.csv"
OUT = ROOT / "analyses/grupo5/outputs"
OUT.mkdir(parents=True, exist_ok=True)
PROTOCOL_PATH = OUT / "protocol.json"
SEED = 42
np.random.seed(SEED)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def load_data():
    df = pd.read_csv(BASE_PATH, parse_dates=["DATE"]).sort_values("DATE").reset_index(drop=True)
    raw_path = ROOT / "bases/grupo5-tratamento/gold_daily_prices.csv"
    raw = pd.read_csv(raw_path, sep=None, engine="python", parse_dates=["DATE"])
    raw["EXPECTED_TARGET"] = pd.to_numeric(raw["VALUE"], errors="coerce").shift(-1)
    raw["TARGET_DATE"] = raw["DATE"].shift(-1)
    target_lookup = raw[["DATE", "TARGET_DATE", "EXPECTED_TARGET"]]
    df = df.merge(target_lookup, on="DATE", how="left", validate="one_to_one")
    assert df["DATE"].is_monotonic_increasing and not df["DATE"].duplicated().any()
    assert "TARGET_UP" not in df.columns and not df.isna().any().any()
    assert np.allclose(df["TARGET"], df["EXPECTED_TARGET"])
    return df.drop(columns="EXPECTED_TARGET")



In [2]:
def load_protocol(require_data_decisions=False):
    if not PROTOCOL_PATH.exists():
        raise RuntimeError("DECISÃO NECESSÁRIA: execute 01_documentacao_bases.ipynb primeiro.")
    protocol = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
    if protocol.get("protocol_version") != 2:
        raise RuntimeError("Protocolo desatualizado: execute novamente 01_documentacao_bases.ipynb.")
    if protocol["dataset_sha256"] != sha256(BASE_PATH):
        raise RuntimeError("A base congelada não corresponde ao hash do protocolo.")
    if require_data_decisions:
        missing = [k for k in ("source", "unit", "cleaning", "features", "seasonal_period") if not protocol["decisions"].get(k)]
        if missing:
            raise RuntimeError(f"DECISÃO NECESSÁRIA: complete as decisões {missing} antes de modelar.")
    return protocol

def save_protocol(protocol):
    PROTOCOL_PATH.write_text(json.dumps(protocol, indent=2, ensure_ascii=False), encoding="utf-8")

def origins(protocol, split):
    return protocol["origins"][split]

def target_dates(df, origin_ids):
    return df["TARGET_DATE"].iloc[origin_ids].to_numpy()


## 2. Conceitos principais do PLS

In [3]:
from IPython.display import Markdown, display

protocol = load_protocol(require_data_decisions=True)
n_components = protocol["decisions"]["accepted_parameters"]["PLS_Regression"]["n_components"]
n_features = len(protocol["decisions"]["features"])

display(
    Markdown(
        f"""
## PLS Regression aplicado ao ouro

PLS cria componentes latentes supervisionados. Cada componente é uma combinação linear das features escolhida para representar a covariância entre os preditores e o preço do próximo pregão.

Neste experimento foram selecionados **{n_components} componentes entre {n_features} features** exclusivamente pelo menor MAE de validação walk-forward. O teste final não alterou essa escolha.

### Por que o método é pertinente

- preço atual, lags e médias móveis são fortemente correlacionados;
- PLS consegue trabalhar com multicolinearidade ao projetar as features em componentes;
- a padronização evita que escalas maiores dominem a construção dos componentes;
- o modelo continua linear e pode extrapolar, ao contrário de uma floresta de árvores.

### Hipóteses e limitações

- componentes descrevem associação preditiva, não causalidade econômica;
- coeficientes podem mudar quando features correlacionadas são adicionadas ou removidas;
- muitos componentes aproximam o PLS de uma regressão linear com pouca redução dimensional;
- mudança de regime entre validação e teste pode alterar o ranking;
- importância por permutação em features correlacionadas tende a dividir importância entre variáveis semelhantes.
"""
    )
)



## PLS Regression aplicado ao ouro

PLS cria componentes latentes supervisionados. Cada componente é uma combinação linear das features escolhida para representar a covariância entre os preditores e o preço do próximo pregão.

Neste experimento foram selecionados **11 componentes entre 12 features** exclusivamente pelo menor MAE de validação walk-forward. O teste final não alterou essa escolha.

### Por que o método é pertinente

- preço atual, lags e médias móveis são fortemente correlacionados;
- PLS consegue trabalhar com multicolinearidade ao projetar as features em componentes;
- a padronização evita que escalas maiores dominem a construção dos componentes;
- o modelo continua linear e pode extrapolar, ao contrário de uma floresta de árvores.

### Hipóteses e limitações

- componentes descrevem associação preditiva, não causalidade econômica;
- coeficientes podem mudar quando features correlacionadas são adicionadas ou removidas;
- muitos componentes aproximam o PLS de uma regressão linear com pouca redução dimensional;
- mudança de regime entre validação e teste pode alterar o ranking;
- importância por permutação em features correlacionadas tende a dividir importância entre variáveis semelhantes.


## Interpretação

O PLS cria componentes orientados pela covariância com o alvo. Ele é útil quando as features são correlacionadas, mas depende de escala adequada e relações aproximadamente lineares.

## Artefatos

Arquivos usados ou produzidos nesta etapa:

- `outputs/PLS_technical_summary.md`
- `outputs/hyperparameters.csv`